<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-III/blob/main/Practicando_LLM/practica_mpg_groq_alumnos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Práctica: Predicción de Consumo (MPG) con Limpieza de Datos vía LLM

¡Bienvenidos/as! Este notebook replica el pipeline que vimos en clase (Groq + pandas + EDA + ML + Deep Learning), pero ahora sobre un dataset **real y conocido**: **Auto MPG** (consumo de combustible de autos de los años 70-80).

Completá las celdas marcadas con `# TODO`. Vas a aplicar:

1. 🤖 Normalización de texto sucio con la API de Groq (LLM)
2. 🧹 Data Wrangling
3. 📊 EDA (Análisis Exploratorio de Datos)
4. 📈 Estadística (correlación)
5. 🤖 Modelos: Regresión Lineal, Random Forest y Red Neuronal (Keras)
6. 🏆 Comparación de modelos

> 💡 **Tip:** este dataset tiene nombres de auto con marcas mal escritas *de verdad* (no simuladas) — es un caso real de "dirty data". Vas a ver marcas como `vw`, `volkswagen`, `chevroelt`, `toyouta`, `maxda`... el mismo problema que resolvimos en clase con `normalizar_marca_con_groq`.


## 📦 1. Setup

In [ ]:
!pip install -q groq pandas scikit-learn tensorflow matplotlib seaborn

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Setup listo.")

## 🔎 2. Cargando y explorando el dataset

Usamos el dataset **Auto MPG**, incluido en seaborn.

In [ ]:
df = sns.load_dataset('mpg')
df.head()

In [ ]:
# TODO: mostrá la forma (shape) del dataset y los tipos de datos (dtypes) de cada columna



## 🏷️ 3. Extrayendo la marca del auto

La columna `name` tiene el nombre completo del auto, ej: `"chevrolet chevelle malibu"`. La marca es la primera palabra.

In [ ]:
# TODO: creá una columna nueva 'marca_original' con la primera palabra de 'name'
# Pista: df['name'].str.split().str[0]
df['marca_original'] = ____

df[['name', 'marca_original']].head()

### 🔎 ¿Cuántas marcas únicas hay?

In [ ]:
# TODO: usá value_counts() sobre 'marca_original' y guardalo en conteo_marcas
conteo_marcas = ____

print(f"🔎 Marcas únicas encontradas: {len(conteo_marcas)} (sobre {len(df)} filas totales)\n")
print(conteo_marcas)

Vas a notar marcas repetidas con distinta escritura (ej: `vw` y `volkswagen`, `chevrolet` y `chevroelt`, `toyota` y `toyouta`, `maxda` en vez de `mazda`). Esto es 100% real, no lo simulamos.

## 🤖 4. Normalización con la API de Groq

Necesitás una API key de Groq guardada en Colab (⚙️ Secrets → `groq_api`). Si no la tenés, generá una gratis en https://console.groq.com

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('groq_api'))
print("🔑 Cliente de Groq inicializado correctamente.")

### ✍️ Completá la función de normalización

Recordá el patrón visto en clase: el prompt le tiene que pedir al modelo que devuelva **ÚNICAMENTE** el nombre correcto de la marca, sin explicaciones extra ni puntuación.

In [ ]:
def normalizar_marca_con_groq(nombre_sucio: str) -> str:
    '''Le pide a un LLM (vía Groq) que devuelva el nombre correcto de una marca de auto.'''

    # TODO: escribí el prompt. Tiene que:
    #   - actuar como un normalizador experto en marcas de autos
    #   - pedir SOLO el nombre correcto de la marca, sin texto adicional
    #   - incluir la variable nombre_sucio dentro del prompt
    prompt = f'''
    ____
    '''

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    # TODO: extraé el texto de la respuesta del modelo y devolvelo limpio (sin espacios ni saltos de línea extra)
    return ____

### 🔁 Aplicando `.map()` sobre las marcas únicas

Recordá: `.map()` recorre cada valor de la Serie y le aplica la función — se le pasa **sin paréntesis**.

In [ ]:
marcas_unicas = conteo_marcas.index.to_series()

print(f"🤖 Consultando la API de Groq para {len(marcas_unicas)} marcas únicas...")

# TODO: aplicá normalizar_marca_con_groq a marcas_unicas usando .map()
marcas_normalizadas = ____

print("\n✅ ¡Normalización terminada! Mapeo resultante:")
print(marcas_normalizadas)

### 🧩 Aplicando el mapeo a todo el dataset

`marcas_normalizadas` es una Serie indexada por la marca *sucia*: es, en la práctica, una tabla de traducción.

In [ ]:
# TODO: creá 'marca_limpia' mapeando 'marca_original' con marcas_normalizadas
# no te olvides del .fillna() para las marcas que no hayan matcheado
df['marca_limpia'] = ____

print(f"📊 Marcas únicas antes de normalizar: {df['marca_original'].nunique()}")
print(f"📊 Marcas únicas después de normalizar: {df['marca_limpia'].nunique()}")
df[['marca_original', 'marca_limpia']].drop_duplicates().sort_values('marca_original').head(15)

## 🧹 5. Data Wrangling

Este dataset tiene un problema clásico: la columna `horsepower` tiene valores faltantes marcados con el string `"?"` en vez de `NaN`.

In [ ]:
# TODO: reemplazá los "?" en 'horsepower' por NaN
# Pista: pd.to_numeric(df['horsepower'], errors='coerce')
df['horsepower'] = ____

# TODO: contá cuántos nulos quedaron por columna
print(____)

In [ ]:
# TODO: eliminá las filas con nulos (dropna) y guardá el resultado en df
df = ____

print(f"Filas restantes: {len(df)}")

In [ ]:
# TODO: convertí 'marca_limpia' y 'origin' a tipo category, y 'model_year' a int
____

df.dtypes

## 📊 6. EDA — Análisis Exploratorio de Datos

### ⛽ Distribución de mpg y del peso (ejemplo ya resuelto)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['mpg'], kde=True, ax=axes[0], color='mediumorchid')
axes[0].set_title('⛽ Distribución de mpg (millas por galón)')

sns.histplot(df['weight'], kde=True, ax=axes[1], color='teal')
axes[1].set_title('⚖️ Distribución del peso')

plt.tight_layout()
plt.show()

### 🚙 Ahora te toca a vos

In [ ]:
# TODO: hacé un boxplot de 'mpg' según 'marca_limpia', ordenado por mediana (de mayor a menor)
# Pista: fijate cómo se armaba 'orden_marcas' en el notebook de clase con groupby().median().sort_values()



In [ ]:
# TODO: hacé un scatterplot de 'horsepower' (eje x) vs 'mpg' (eje y), coloreado (hue) por 'origin'



## 📈 7. Estadística: Matriz de correlación

In [ ]:
# TODO: armá la matriz de correlación de Pearson entre las variables numéricas
# (mpg, cylinders, displacement, horsepower, weight, acceleration, model_year)
# y mostrala con un heatmap (sns.heatmap, cmap='coolwarm', vmin=-1, vmax=1, annot=True)

num_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
corr = ____



## 🛠️ 8. Preparación de datos para Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TODO: armá X (features, con pd.get_dummies para 'marca_limpia' y 'origin', drop_first=True)
# usá como features: cylinders, displacement, horsepower, weight, acceleration, model_year, marca_limpia, origin
# y armá y = df['mpg']
X = ____
y = ____

print(f"🧩 Features usadas ({X.shape[1]})")

In [ ]:
# TODO: hacé el train_test_split (test_size=0.2, random_state=20)
X_train, X_test, y_train, y_test = ____

print(f"🏋️ Train: {X_train.shape[0]} autos | 🧪 Test: {X_test.shape[0]} autos")

In [ ]:
# TODO: escalá X_train y X_test con StandardScaler (fit_transform en train, transform en test)
scaler_X = ____
X_train_s = ____
X_test_s = ____

## 📐 9. Regresión Lineal Múltiple

Empezamos con el modelo más simple e interpretable.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: entrená una LinearRegression con X_train/y_train y prediní sobre X_test
lr = ____
____  # fit
pred_lr = ____

In [ ]:
# TODO: calculá RMSE, MAE y R2 para pred_lr, e imprimilos
# Pista: RMSE = mean_squared_error(y_test, pred) ** 0.5

rmse_lr = ____
mae_lr = ____
r2_lr = ____

print("📐 Regresión Lineal — métricas en test:")
print(f"   RMSE: {rmse_lr:.2f}")
print(f"   MAE:  {mae_lr:.2f}")
print(f"   R²:   {r2_lr:.4f}")

## 🌲 10. Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: entrená un RandomForestRegressor (n_estimators=300, random_state=20, n_jobs=-1) y prediní sobre X_test
rf = ____
____  # fit
pred_rf = ____

In [ ]:
# TODO: calculá RMSE, MAE y R2 para pred_rf (mismo criterio que con la regresión lineal)
rmse_rf = ____
mae_rf = ____
r2_rf = ____

print("🌲 Random Forest — métricas en test:")
print(f"   RMSE: {rmse_rf:.2f}")
print(f"   MAE:  {mae_rf:.2f}")
print(f"   R²:   {r2_rf:.4f}")

## 🧠 11. Red Neuronal (Deep Learning)

Recordá la arquitectura vista en clase: 2 capas ocultas con activación `relu`, y `EarlyStopping` monitoreando `val_loss`.

In [ ]:
import tensorflow as tf
tf.random.set_seed(20)

# TODO: armá un Sequential con:
#  - Input(shape=(X_train_s.shape[1],))
#  - Dense(64, activation='relu')
#  - Dense(32, activation='relu')
#  - Dense(1)  (capa de salida, sin activación: es regresión)
nn_model = ____

nn_model.summary()

In [ ]:
# TODO: compilá el modelo (optimizer='adam', loss='mse', metrics=['mae'])
____

# TODO: definí un EarlyStopping (monitor='val_loss', patience=15, restore_best_weights=True)
early_stop = ____

In [ ]:
# TODO: entrená con validation_split=0.2, epochs=200, batch_size=16, usando early_stop como callback
history = ____

print(f"🏁 Entrenamiento terminado en {len(history.history['loss'])} épocas")

In [ ]:
# TODO: graficá loss vs val_loss (usá history.history['loss'] y history.history['val_loss'])



In [ ]:
# Nota: acá NO escalamos y (a diferencia del notebook de clase), así que la predicción ya está en unidades reales de mpg
pred_nn = nn_model.predict(X_test_s, verbose=0).flatten()

# TODO: calculá RMSE, MAE y R2 para pred_nn
rmse_nn = ____
mae_nn = ____
r2_nn = ____

print("🧠 Red Neuronal — métricas en test:")
print(f"   RMSE: {rmse_nn:.2f}")
print(f"   MAE:  {mae_nn:.2f}")
print(f"   R²:   {r2_nn:.4f}")

## 🏆 12. Comparación final de modelos

In [ ]:
# TODO: armá un DataFrame comparando RMSE, MAE y R2 de los 3 modelos, ordenado por RMSE ascendente
resultados = ____

resultados

In [ ]:
# TODO: graficá con barplots el RMSE y el R2 de los 3 modelos, uno al lado del otro (subplots 1x2)



## ✍️ 13. Conclusiones

Completá con tus propias palabras:

- ¿Qué modelo funcionó mejor en este dataset y por qué creés que fue así?
- ¿Qué variable te pareció más determinante para predecir el consumo (mpg)?
- ¿Qué tan útil fue normalizar las marcas con el LLM comparado con hacerlo a mano, con un dataset de este tamaño?

_(Escribí tu respuesta acá)_
